# VoxCPM2 Dual‑T4 Voice Studio — Final UI

Kaggle-focused Burmese voice studio with two T4 workers, long-form chunking, 12 voice-design profiles, persistent saved voices, remote/direct voice cloning, optional cleanup, Whisper transcription, deterministic speed control, and exact chunk-join pauses.

**Important:** restart the Kaggle session once before switching from the older notebook version so old GPU worker processes do not remain in VRAM.


In [1]:
# Cell 1 — install Voice Studio dependencies
!pip -q install -U \
    voxcpm soundfile kagglehub huggingface_hub \
    yt-dlp gdown faster-whisper noisereduce \
    mega.py-v2 demucs

# Kaggle normally has ffmpeg; install it only when missing.
!which ffmpeg >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq ffmpeg)

print("✓ Voice Studio dependencies installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━

In [2]:
# Cell 2 — download model once to a shared local directory
from huggingface_hub import snapshot_download
import os

MODEL_DIR = "/kaggle/working/VoxCPM2"

if not os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")):
    print("Downloading VoxCPM2...")
    snapshot_download(
        repo_id="openbmb/VoxCPM2",
        local_dir=MODEL_DIR,
    )
else:
    print("✓ VoxCPM2 already exists in", MODEL_DIR)

print("✓ Model directory:", MODEL_DIR)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

✓ Model directory: /kaggle/working/VoxCPM2


In [3]:
%%writefile /kaggle/working/voxcpm_dual_worker.py
import os
import random
import inspect
import traceback

def worker_main(physical_gpu_id, model_dir, task_q, result_q, optimize=False):
    # One physical T4 is exposed to each child. Inside the child, "cuda" is correct.
    os.environ["CUDA_VISIBLE_DEVICES"] = str(physical_gpu_id)

    try:
        import numpy as np
        import torch
        import soundfile as sf
        from voxcpm import VoxCPM

        result_q.put({
            "status": "loading",
            "gpu": physical_gpu_id,
        })

        model = VoxCPM.from_pretrained(
            model_dir,
            load_denoiser=False,
            optimize=bool(optimize),
            device="cuda",
        )

        try:
            sample_rate = int(model.tts_model.sample_rate)
        except Exception:
            sample_rate = 48000

        # Detect the parameters accepted by the *installed* VoxCPM build.
        try:
            sig = inspect.signature(model._generate)
            params = sig.parameters
            accepts_any_kwarg = any(
                p.kind == inspect.Parameter.VAR_KEYWORD
                for p in params.values()
            )
            supported = set(params.keys())
        except Exception:
            accepts_any_kwarg = True
            supported = set()

        result_q.put({
            "status": "ready",
            "gpu": physical_gpu_id,
            "sample_rate": sample_rate,
            "supported_params": sorted(supported),
        })

    except Exception as e:
        result_q.put({
            "status": "startup_error",
            "gpu": physical_gpu_id,
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        return

    while True:
        task = task_q.get()

        if task is None:
            break

        batch_id = task["batch_id"]
        task_id = task["task_id"]
        chunk_index = task.get("chunk_index", 0)
        output_path = task["output_path"]
        kwargs = dict(task["kwargs"])

        try:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # ---------------------------------------------------------
            # VERSION-COMPATIBILITY LAYER
            # ---------------------------------------------------------
            # Recent VoxCPM builds accept seed= in _generate().
            # The Kaggle build in this user's runtime does not.
            #
            # If seed isn't supported, set Python/NumPy/PyTorch RNGs
            # manually, then REMOVE seed before model.generate().
            # ---------------------------------------------------------
            requested_seed = kwargs.get("seed", None)

            if (
                requested_seed is not None
                and not accepts_any_kwarg
                and "seed" not in supported
            ):
                requested_seed = int(requested_seed)

                random.seed(requested_seed)
                np.random.seed(requested_seed % (2**32 - 1))
                torch.manual_seed(requested_seed)

                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(requested_seed)

                kwargs.pop("seed", None)

            # Filter every unsupported UI option too. This makes the notebook
            # resilient if Kaggle has an older VoxCPM than the current source.
            dropped_kwargs = []

            if not accepts_any_kwarg:
                for key in list(kwargs.keys()):
                    if key not in supported:
                        dropped_kwargs.append(key)
                        kwargs.pop(key, None)

            wav = model.generate(**kwargs)

            sf.write(
                output_path,
                wav,
                sample_rate,
                subtype="PCM_16",
            )

            result_q.put({
                "status": "ok",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "output_path": output_path,
                "sample_rate": sample_rate,
                "samples": int(len(wav)),
                "duration": float(len(wav) / sample_rate),
                "dropped_kwargs": dropped_kwargs,
            })

        except Exception as e:
            result_q.put({
                "status": "error",
                "batch_id": batch_id,
                "task_id": task_id,
                "chunk_index": chunk_index,
                "gpu": physical_gpu_id,
                "error": repr(e),
                "traceback": traceback.format_exc(),
            })


Writing /kaggle/working/voxcpm_dual_worker.py


In [4]:
# ============================================================
# CELL 4 — VoxCPM2 Dual-T4 Voice Studio v6
# Burmese-safe long-form generation + continuous chunk streaming
#
# Assumes:
#   • MODEL_DIR exists (normally /kaggle/working/VoxCPM2)
#   • Cell 3 created /kaggle/working/voxcpm_dual_worker.py
#     containing worker_main(...)
# ============================================================

import os
import re
import sys
import gc
import json
import glob
import time
import uuid
import base64
import shutil
import queue as pyqueue
import threading
import subprocess
import unicodedata
from pathlib import Path
from multiprocessing import get_context
from urllib.parse import urlparse

import numpy as np
import soundfile as sf
import ipywidgets as widgets

from IPython.display import (
    display,
    Audio,
    FileLink,
    HTML,
    Javascript,
    clear_output,
)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

MODEL_DIR = globals().get(
    "MODEL_DIR",
    "/kaggle/working/VoxCPM2",
)

VOICE_DATASET_HANDLE_DEFAULT = "cyawzin/voxcpm-voice-library"

# Keep torch.compile off for T4 stability.
WORKER_OPTIMIZE = False
WORKER_STARTUP_TIMEOUT = 900

LIBRARY_ROOT = Path("/kaggle/working/voxcpm_voice_library")
VOICE_DIR = LIBRARY_ROOT / "voices"
GENERATED_DIR = LIBRARY_ROOT / "generated"
IMPORT_DIR = LIBRARY_ROOT / "imports"
COOKIE_DIR = LIBRARY_ROOT / "cookies"
META_FILE = LIBRARY_ROOT / "library.json"

for _p in (
    LIBRARY_ROOT,
    VOICE_DIR,
    GENERATED_DIR,
    IMPORT_DIR,
    COOKIE_DIR,
):
    _p.mkdir(parents=True, exist_ok=True)

LIBRARY_LOCK = threading.Lock()

# ------------------------------------------------------------
# VOICE DESIGN PROFILES
# ------------------------------------------------------------

VOICE_PROFILES = {
    "u_aung": {
        "name": "👨 U Aung — Deep Calm Male",
        "design": (
            "An adult Burmese man in his late thirties to mid forties. "
            "A naturally low baritone voice with warm chest resonance, "
            "calm confidence, mature presence, natural Burmese rhythm, "
            "clean articulation and relaxed medium-slow pacing."
        ),
    },
    "ko_min": {
        "name": "👨 Ko Min — Friendly Young Male",
        "design": (
            "A Burmese man in his mid to late twenties with a friendly, "
            "approachable medium-pitched voice. Warm, relaxed, youthful, "
            "conversational and natural with smooth phrasing."
        ),
    },
    "u_htet": {
        "name": "👴 U Htet — Wise Older Male",
        "design": (
            "A Burmese man around sixty with a mature deep voice, gentle age "
            "texture and warmth. Patient, reassuring storyteller delivery "
            "with measured pacing and thoughtful pauses."
        ),
    },
    "ko_zaw": {
        "name": "👨 Ko Zaw — Energetic Presenter",
        "design": (
            "A Burmese man around thirty with a clear, energetic and confident "
            "speaking voice. Lively but controlled, crisp articulation and "
            "slightly quick pacing without shouting."
        ),
    },
    "u_kyaw": {
        "name": "👔 U Kyaw — Formal Professional Male",
        "design": (
            "A professional Burmese man in his forties with a composed, polished "
            "and trustworthy voice. Precise articulation, stable volume, "
            "confident phrasing and a measured medium pace."
        ),
    },
    "daw_may": {
        "name": "👩 Daw May — Warm Adult Female",
        "design": (
            "A Burmese woman in her late thirties to mid forties with a warm, "
            "gentle, naturally feminine voice. Caring, grounded and trustworthy "
            "with soft but clear articulation."
        ),
    },
    "ma_su": {
        "name": "👩 Ma Su — Bright Young Female",
        "design": (
            "A Burmese woman in her early to mid twenties with a youthful, "
            "bright and naturally pleasant voice. Cheerful, friendly and "
            "expressive without sounding childish."
        ),
    },
    "daw_thida": {
        "name": "👩 Daw Thida — Elegant Mature Female",
        "design": (
            "A mature Burmese woman with a smooth, composed and elegant voice. "
            "Refined, confident, warm and professionally controlled."
        ),
    },
    "ma_nandar": {
        "name": "👩 Ma Nandar — Soft Emotional Female",
        "design": (
            "A Burmese woman in her late twenties to early thirties with a "
            "soft, intimate and emotionally expressive voice. Personal, sincere "
            "and clearly audible with gentle emotional variation."
        ),
    },
    "news_presenter": {
        "name": "📰 Burmese News Presenter",
        "design": (
            "A professional adult Burmese news presenter with a neutral, "
            "trustworthy and highly intelligible broadcast voice. Precise "
            "pronunciation, stable volume and controlled sentence emphasis."
        ),
    },
    "documentary": {
        "name": "🎥 Deep Documentary Narrator",
        "design": (
            "A mature Burmese documentary narrator with a naturally deep, "
            "resonant voice. Intelligent, observant and serious with controlled "
            "low-register resonance and subtle suspense."
        ),
    },
    "movie_recap": {
        "name": "🎬 Fast Movie Recap Narrator",
        "design": (
            "A young adult Burmese male movie-recap narrator with a natural, "
            "energetic and highly engaging voice. Speak noticeably faster than "
            "normal conversation while preserving clean pronunciation and "
            "strong forward momentum."
        ),
    },
}

DELIVERY_PRESETS = {
    "Natural / keep cloned delivery": "",
    "Warm & conversational": (
        "warm, natural, conversational delivery with subtle human pauses"
    ),
    "Calm & serious": (
        "calm, serious, controlled delivery with clear articulation"
    ),
    "Friendly": (
        "friendly, approachable, relaxed conversational delivery"
    ),
    "Professional / formal": (
        "professional, formal, polished delivery with precise articulation"
    ),
    "Documentary": (
        "deep, restrained documentary narration with subtle suspense"
    ),
    "Movie recap": (
        "fast-paced engaging movie recap delivery with strong momentum "
        "and clean articulation"
    ),
    "Excited": (
        "energetic and excited but controlled, clear and never shouting"
    ),
    "Soft / emotional": (
        "soft, intimate and sincere delivery with gentle emotional variation"
    ),
}

# ------------------------------------------------------------
# RESTORE SAVED LIBRARY FROM AN ATTACHED KAGGLE DATASET
# ------------------------------------------------------------

def restore_attached_library_if_needed():
    if META_FILE.exists():
        return None

    candidates = glob.glob(
        "/kaggle/input/**/library.json",
        recursive=True,
    )

    for meta in candidates:
        src_root = Path(meta).parent
        src_voice_dir = src_root / "voices"

        if src_voice_dir.exists():
            shutil.copytree(
                src_root,
                LIBRARY_ROOT,
                dirs_exist_ok=True,
            )
            return str(src_root)

    return None


restored_from = restore_attached_library_if_needed()

# ------------------------------------------------------------
# VOICE LIBRARY
# ------------------------------------------------------------

def empty_library():
    return {
        "version": 6,
        "active_voice_id": None,
        "voices": {},
    }


def load_library():
    if not META_FILE.exists():
        return empty_library()

    try:
        data = json.loads(
            META_FILE.read_text(encoding="utf-8")
        )
        data.setdefault("version", 6)
        data.setdefault("active_voice_id", None)
        data.setdefault("voices", {})
        return data
    except Exception:
        return empty_library()


library = load_library()


def save_library():
    with LIBRARY_LOCK:
        META_FILE.write_text(
            json.dumps(
                library,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )


def voice_path(voice_id):
    return (
        LIBRARY_ROOT
        / library["voices"][voice_id]["file"]
    )


def valid_voice_ids():
    result = []

    for vid, entry in library["voices"].items():
        p = LIBRARY_ROOT / entry.get("file", "")

        if p.exists():
            result.append(vid)

    return result

# ------------------------------------------------------------
# BURMESE-SAFE TEXT CHUNKING
# ------------------------------------------------------------

BOUNDARIES = [
    "။",
    "!",
    "?",
    "…",
    ".",
    "၊",
    ",",
    ";",
    ":",
    "\n",
    " ",
]


def sanitize_tts_text(text):
    """
    Clean invisible/control characters without running WeText normalization.
    """
    text = unicodedata.normalize(
        "NFC",
        str(text or ""),
    )

    # Remove BOM / zero-width characters that can break text parsing.
    text = (
        text
        .replace("\ufeff", "")
        .replace("\u200b", "")
        .replace("\u200c", "")
        .replace("\u200d", "")
        .replace("\u2060", "")
    )

    # Convert most control chars to spaces, but preserve newlines first.
    text = re.sub(
        r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]",
        " ",
        text,
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


def has_speakable_content(text):
    """
    Myanmar base letters are Unicode Letter categories.
    We also permit numbers.
    """
    for ch in str(text or ""):
        category = unicodedata.category(ch)

        if category.startswith("L"):
            return True

        if category.startswith("N"):
            return True

    return False


def split_long_text(text, max_chars=220):
    text = sanitize_tts_text(text)

    if not text:
        return []

    chunks = []
    remaining = text

    while len(remaining) > int(max_chars):
        window = remaining[: int(max_chars) + 1]

        best = -1

        for mark in BOUNDARIES:
            pos = window.rfind(mark)

            if pos > best:
                best = pos

        # Avoid a tiny chunk caused by very old punctuation.
        if best < int(max_chars * 0.55):
            best = int(max_chars)
        else:
            best += 1

        piece = remaining[:best].strip()

        if piece:
            chunks.append(piece)

        remaining = remaining[best:].strip()

    if remaining:
        chunks.append(remaining)

    return chunks


def clean_chunks_for_tts(chunks):
    """
    Remove empty/punctuation-only fragments and merge punctuation-only
    fragments into the previous chunk when useful.
    """
    cleaned = []

    for chunk in chunks:
        chunk = sanitize_tts_text(chunk)

        if not chunk:
            continue

        if not has_speakable_content(chunk):
            if cleaned:
                cleaned[-1] = (
                    cleaned[-1] + " " + chunk
                ).strip()
            continue

        cleaned.append(chunk)

    return cleaned

# ------------------------------------------------------------
# AUDIO HELPERS
# ------------------------------------------------------------

def _mono_float32(wav):
    wav = np.asarray(
        wav,
        dtype=np.float32,
    )

    if wav.ndim == 1:
        return wav

    if wav.ndim == 2:
        return wav.mean(
            axis=1,
            dtype=np.float32,
        )

    return wav.reshape(-1).astype(
        np.float32
    )


# Alias so either helper name works.
mono_float32 = _mono_float32


def trim_edge_silence(
    wav,
    sr,
    threshold_db=-42.0,
    keep_ms=25,
):
    """
    Trim only leading/trailing silence.
    Interior pauses remain untouched.
    """
    wav = _mono_float32(wav)

    if len(wav) == 0:
        return wav

    peak = float(
        np.max(
            np.abs(wav)
        )
    )

    if peak < 1e-7:
        return wav

    threshold = (
        peak
        * (10.0 ** (float(threshold_db) / 20.0))
    )

    active = np.flatnonzero(
        np.abs(wav) >= threshold
    )

    if active.size == 0:
        return wav

    keep = int(
        sr
        * float(keep_ms)
        / 1000.0
    )

    start = max(
        0,
        int(active[0]) - keep,
    )

    end = min(
        len(wav),
        int(active[-1]) + keep + 1,
    )

    return wav[start:end]


def ffmpeg_atempo(
    input_path,
    output_path,
    speed,
):
    """
    Pitch-preserving speed adjustment.
    UI range stays inside FFmpeg atempo's direct 0.5-2.0 range.
    """
    speed = float(speed)

    if abs(speed - 1.0) < 0.001:
        shutil.copy2(
            input_path,
            output_path,
        )
        return str(output_path)

    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
        "-i",
        str(input_path),
        "-filter:a",
        f"atempo={speed:.4f}",
        "-c:a",
        "pcm_s16le",
        str(output_path),
    ]

    subprocess.run(
        cmd,
        check=True,
    )

    return str(output_path)


def prepare_stream_chunk(
    raw_path,
    output_path,
    speed=1.0,
    trim_edges=True,
    trim_db=-42.0,
):
    """
    Process one completed VoxCPM chunk exactly once.
    The same processed chunk is used for:
      • individual preview
      • continuous stream queue
      • final combined WAV
    """
    raw_path = Path(raw_path)
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    working = raw_path
    temp_speed = None

    try:
        if abs(float(speed) - 1.0) >= 0.001:
            temp_speed = output_path.with_name(
                output_path.stem
                + "_speedtmp.wav"
            )

            ffmpeg_atempo(
                str(raw_path),
                str(temp_speed),
                float(speed),
            )

            working = temp_speed

        wav, sr = sf.read(
            str(working),
            dtype="float32",
            always_2d=False,
        )

        wav = _mono_float32(wav)

        if bool(trim_edges):
            wav = trim_edge_silence(
                wav,
                sr,
                threshold_db=float(trim_db),
                keep_ms=25,
            )

        sf.write(
            str(output_path),
            wav,
            int(sr),
            subtype="PCM_16",
        )

        return (
            str(output_path),
            int(sr),
            int(len(wav)),
        )

    finally:
        if temp_speed is not None:
            try:
                Path(temp_speed).unlink(
                    missing_ok=True
                )
            except Exception:
                pass


def concatenate_prepared_chunks(
    prepared_paths,
    output_path,
    pause_ms=120,
):
    arrays = []
    target_sr = None

    for i, path in enumerate(
        prepared_paths
    ):
        wav, sr = sf.read(
            str(path),
            dtype="float32",
            always_2d=False,
        )

        wav = _mono_float32(wav)
        sr = int(sr)

        if target_sr is None:
            target_sr = sr

        elif sr != target_sr:
            raise RuntimeError(
                f"Sample-rate mismatch: "
                f"{sr} != {target_sr}"
            )

        arrays.append(wav)

        if (
            i != len(prepared_paths) - 1
            and int(pause_ms) > 0
        ):
            arrays.append(
                np.zeros(
                    int(
                        target_sr
                        * int(pause_ms)
                        / 1000.0
                    ),
                    dtype=np.float32,
                )
            )

    if not arrays:
        raise RuntimeError(
            "No prepared chunks "
            "were available."
        )

    final = np.concatenate(arrays)

    sf.write(
        str(output_path),
        final,
        target_sr or 48000,
        subtype="PCM_16",
    )

    return (
        str(output_path),
        target_sr or 48000,
        int(len(final)),
    )


def make_stream_segment_mp3(
    prepared_wav,
    output_mp3,
    pause_ms,
    add_pause=True,
):
    """
    Make a small 64-kbps MP3 browser segment.

    We append the requested join pause to every non-final stream segment.
    The final downloadable file is still WAV.
    """
    prepared_wav = Path(
        prepared_wav
    )

    output_mp3 = Path(
        output_mp3
    )

    temp_wav = (
        output_mp3
        .with_suffix(
            ".segment.wav"
        )
    )

    wav, sr = sf.read(
        str(prepared_wav),
        dtype="float32",
        always_2d=False,
    )

    wav = _mono_float32(wav)
    sr = int(sr)

    if (
        add_pause
        and int(pause_ms) > 0
    ):
        silence = np.zeros(
            int(
                sr
                * int(pause_ms)
                / 1000.0
            ),
            dtype=np.float32,
        )

        wav = np.concatenate(
            [
                wav,
                silence,
            ]
        )

    sf.write(
        str(temp_wav),
        wav,
        sr,
        subtype="PCM_16",
    )

    try:
        cmd = [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            str(temp_wav),
            "-vn",
            "-ac",
            "1",
            "-codec:a",
            "libmp3lame",
            "-b:a",
            "64k",
            str(output_mp3),
        ]

        subprocess.run(
            cmd,
            check=True,
        )

    finally:
        try:
            temp_wav.unlink(
                missing_ok=True
            )
        except Exception:
            pass

    return str(output_mp3)


def file_as_data_url(
    path,
    mime="audio/mpeg",
):
    encoded = base64.b64encode(
        Path(path).read_bytes()
    ).decode("ascii")

    return (
        f"data:{mime};base64,"
        f"{encoded}"
    )


def normalize_media_to_wav(
    source_path,
    output_path,
    start_seconds=0.0,
    max_seconds=20.0,
):
    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel",
        "error",
    ]

    if float(start_seconds) > 0:
        cmd += [
            "-ss",
            str(float(start_seconds)),
        ]

    cmd += [
        "-i",
        str(source_path),
    ]

    if float(max_seconds) > 0:
        cmd += [
            "-t",
            str(float(max_seconds)),
        ]

    cmd += [
        "-vn",
        "-ac",
        "1",
        "-ar",
        "48000",
        "-c:a",
        "pcm_s16le",
        str(output_path),
    ]

    subprocess.run(
        cmd,
        check=True,
    )

    return str(output_path)


def reduce_residual_noise(
    input_path,
    output_path,
    amount=0.70,
):
    import noisereduce as nr

    wav, sr = sf.read(
        str(input_path),
        dtype="float32",
        always_2d=False,
    )

    wav = _mono_float32(wav)

    cleaned = nr.reduce_noise(
        y=wav,
        sr=int(sr),
        stationary=False,
        prop_decrease=float(amount),
    )

    sf.write(
        str(output_path),
        cleaned,
        int(sr),
        subtype="PCM_16",
    )

    return str(output_path)


def extract_vocals_demucs(
    input_path,
    job_dir,
):
    """
    Runs Demucs on CPU so both T4s remain available for VoxCPM2.
    """
    out_root = (
        Path(job_dir)
        / "demucs"
    )

    cmd = [
        sys.executable,
        "-m",
        "demucs.separate",
        "-d",
        "cpu",
        "-n",
        "htdemucs",
        "--two-stems=vocals",
        "--float32",
        "--shifts",
        "0",
        "--overlap",
        "0.10",
        "-o",
        str(out_root),
        str(input_path),
    ]

    subprocess.run(
        cmd,
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    candidates = list(
        out_root.glob(
            "**/vocals.wav"
        )
    )

    if not candidates:
        raise RuntimeError(
            "Demucs finished but "
            "vocals.wav was not found."
        )

    return str(candidates[0])


def transcribe_reference(
    wav_path,
    model_name="large-v3-turbo",
    language="my",
):
    from faster_whisper import WhisperModel

    model = WhisperModel(
        model_name,
        device="cpu",
        compute_type="int8",
        cpu_threads=max(
            2,
            min(
                8,
                os.cpu_count() or 4,
            ),
        ),
    )

    lang = (
        None
        if language == "auto"
        else language
    )

    segments, info = model.transcribe(
        str(wav_path),
        language=lang,
        task="transcribe",
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=True,
    )

    text_parts = []

    for seg in segments:
        t = (
            seg.text or ""
        ).strip()

        if t:
            text_parts.append(t)

    detected = getattr(
        info,
        "language",
        None,
    )

    del model
    gc.collect()

    return (
        " ".join(
            text_parts
        ).strip(),
        detected,
    )

# ------------------------------------------------------------
# UPLOAD / REMOTE MEDIA HELPERS
# ------------------------------------------------------------

def _first_upload(widget):
    value = widget.value

    if not value:
        return None

    if isinstance(
        value,
        (tuple, list),
    ):
        item = value[0]

        return {
            "name": item.get(
                "name",
                "upload.bin",
            ),
            "content": bytes(
                item["content"]
            ),
        }

    if isinstance(
        value,
        dict,
    ):
        first_key = next(
            iter(value)
        )

        item = value[
            first_key
        ]

        name = (
            item
            .get(
                "metadata",
                {},
            )
            .get(
                "name",
                first_key,
            )
        )

        return {
            "name": name,
            "content": bytes(
                item["content"]
            ),
        }

    return None


def save_upload_to_dir(
    upload_widget,
    dest_dir,
    fallback_name="upload.bin",
):
    item = _first_upload(
        upload_widget
    )

    if item is None:
        raise ValueError(
            "No local file "
            "has been uploaded."
        )

    name = Path(
        item["name"]
        or fallback_name
    ).name

    dest = (
        Path(dest_dir)
        / name
    )

    dest.write_bytes(
        item["content"]
    )

    return str(dest)


def save_cookie_upload(
    cookie_widget,
):
    item = _first_upload(
        cookie_widget
    )

    if item is None:
        return None

    path = (
        COOKIE_DIR
        / "cookies.txt"
    )

    path.write_bytes(
        item["content"]
    )

    return str(path)


def download_remote_media(
    url,
    job_dir,
    cookie_file=None,
    user_agent="",
    referer="",
):
    """
    Use only with media the user is permitted to access/download.
    """
    url = (
        url or ""
    ).strip()

    if not url:
        raise ValueError(
            "Remote URL is empty."
        )

    host = (
        urlparse(url)
        .netloc
        .lower()
    )

    job_dir = Path(
        job_dir
    )

    job_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Google Drive
    if (
        "drive.google.com" in host
        or "docs.google.com" in host
    ):
        import gdown

        out = (
            job_dir
            / "drive_source"
        )

        result = gdown.download(
            url=url,
            output=str(out),
            quiet=False,
            fuzzy=True,
        )

        if not result:
            raise RuntimeError(
                "Google Drive "
                "download failed."
            )

        return str(result)

    # MEGA public file link
    if (
        "mega.nz" in host
        or "mega.co.nz" in host
    ):
        from mega import Mega

        mega = Mega()
        client = mega.login()

        before = set(
            job_dir.iterdir()
        )

        result = client.download_url(
            url,
            dest_path=str(
                job_dir
            ),
        )

        if (
            result
            and Path(result).exists()
        ):
            return str(result)

        after = set(
            job_dir.iterdir()
        )

        new_files = [
            p
            for p in (
                after - before
            )
            if p.is_file()
        ]

        if not new_files:
            raise RuntimeError(
                "MEGA download did "
                "not create a file."
            )

        return str(
            max(
                new_files,
                key=lambda p: p.stat().st_size,
            )
        )

    # YouTube / TikTok / direct HTTP / yt-dlp supported sites.
    import yt_dlp

    outtmpl = str(
        job_dir
        / "remote_source.%(ext)s"
    )

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": outtmpl,
        "noplaylist": True,
        "quiet": True,
        "retries": 3,
        "fragment_retries": 3,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "wav",
            }
        ],
    }

    if (
        cookie_file
        and Path(
            cookie_file
        ).exists()
    ):
        ydl_opts[
            "cookiefile"
        ] = str(
            cookie_file
        )

    headers = {}

    if user_agent.strip():
        headers[
            "User-Agent"
        ] = (
            user_agent.strip()
        )

    if referer.strip():
        headers[
            "Referer"
        ] = (
            referer.strip()
        )

    if headers:
        ydl_opts[
            "http_headers"
        ] = headers

    with yt_dlp.YoutubeDL(
        ydl_opts
    ) as ydl:
        ydl.download(
            [url]
        )

    candidates = [
        p
        for p in job_dir.glob(
            "remote_source.*"
        )
        if (
            p.is_file()
            and p.suffix != ".part"
        )
    ]

    if not candidates:
        raise RuntimeError(
            "Remote download did "
            "not produce a media file."
        )

    return str(
        max(
            candidates,
            key=lambda p: p.stat().st_size,
        )
    )

# ------------------------------------------------------------
# DUAL-GPU SERVICE
# ------------------------------------------------------------

sys.path.insert(
    0,
    "/kaggle/working",
)

from voxcpm_dual_worker import worker_main


class DualGPUVoxCPM:
    def __init__(
        self,
        model_dir,
        gpu_ids=(0, 1),
        optimize=False,
        startup_timeout=900,
        heartbeat_seconds=10,
    ):
        self.model_dir = model_dir
        self.gpu_ids = tuple(
            gpu_ids
        )

        self.ctx = get_context(
            "spawn"
        )

        self.task_q = (
            self.ctx.Queue()
        )

        self.result_q = (
            self.ctx.Queue()
        )

        self.procs = []
        self.sample_rate = 48000

        print(
            f"Starting "
            f"{len(self.gpu_ids)} "
            f"VoxCPM2 workers "
            f"(optimize={optimize})..."
        )

        for gpu_id in self.gpu_ids:
            p = self.ctx.Process(
                target=worker_main,
                args=(
                    gpu_id,
                    self.model_dir,
                    self.task_q,
                    self.result_q,
                    optimize,
                ),
                name=(
                    f"VoxCPM2-GPU-"
                    f"{gpu_id}"
                ),
            )

            p.start()
            self.procs.append(p)

            print(
                f"  • worker PID "
                f"{p.pid} → "
                f"physical GPU "
                f"{gpu_id}"
            )

        ready = set()

        deadline = (
            time.time()
            + float(
                startup_timeout
            )
        )

        last_status = 0.0

        while len(ready) < len(
            self.gpu_ids
        ):
            now = time.time()

            for (
                gpu_id,
                proc,
            ) in zip(
                self.gpu_ids,
                self.procs,
            ):
                if (
                    gpu_id not in ready
                    and proc.exitcode
                    is not None
                ):
                    self.shutdown(
                        force=True
                    )

                    raise RuntimeError(
                        f"GPU {gpu_id} worker "
                        f"exited during startup "
                        f"(exit code "
                        f"{proc.exitcode})."
                    )

            if now > deadline:
                statuses = [
                    (
                        f"GPU {gpu}: "
                        f"pid={proc.pid}, "
                        f"alive={proc.is_alive()}, "
                        f"exitcode={proc.exitcode}, "
                        f"ready={gpu in ready}"
                    )
                    for (
                        gpu,
                        proc,
                    ) in zip(
                        self.gpu_ids,
                        self.procs,
                    )
                ]

                self.shutdown(
                    force=True
                )

                raise TimeoutError(
                    "Timed out loading "
                    "VoxCPM2 workers:\n"
                    + "\n".join(
                        statuses
                    )
                )

            try:
                msg = (
                    self.result_q.get(
                        timeout=5
                    )
                )

            except pyqueue.Empty:
                if (
                    now
                    - last_status
                    >= heartbeat_seconds
                ):
                    waiting = [
                        str(gpu)
                        for gpu
                        in self.gpu_ids
                        if gpu
                        not in ready
                    ]

                    print(
                        "  … still loading "
                        "GPU worker(s) "
                        + ", ".join(
                            waiting
                        )
                    )

                    last_status = now

                continue

            status = msg.get(
                "status"
            )

            if status == "loading":
                print(
                    f"  • GPU "
                    f"{msg.get('gpu')} "
                    f"is loading VoxCPM2..."
                )

            elif status == "ready":
                gpu = msg[
                    "gpu"
                ]

                if gpu not in ready:
                    ready.add(
                        gpu
                    )

                    self.sample_rate = msg.get(
                        "sample_rate",
                        self.sample_rate,
                    )

                    print(
                        f"✓ GPU {gpu} "
                        f"ready "
                        f"({self.sample_rate} Hz)"
                    )

            elif status == "startup_error":
                self.shutdown(
                    force=True
                )

                raise RuntimeError(
                    f"GPU "
                    f"{msg.get('gpu')} "
                    f"failed to start:\n"
                    f"{msg.get('error')}\n\n"
                    f"{msg.get('traceback', '')}"
                )

        print(
            "✓ Both VoxCPM2 "
            "workers are ready."
        )

    def run(
        self,
        task_specs,
        on_result=None,
        stop_event=None,
    ):
        """
        Run tasks across both GPU workers, keeping at most one
        in-flight task per GPU.

        Stop prevents new chunks from being scheduled.
        The one or two chunks already running are allowed to finish.

        Returns:
            (sorted_results, stopped)
        """
        if not task_specs:
            return [], False

        batch_id = uuid.uuid4().hex
        total = len(task_specs)
        results = []

        next_index = 0
        in_flight = {}
        stopped = False

        def submit_one(index):
            task_id = f"{batch_id}_{index}"

            payload = dict(task_specs[index])
            payload["batch_id"] = batch_id
            payload["task_id"] = task_id

            in_flight[task_id] = index
            self.task_q.put(payload)

        initial = min(
            len(self.gpu_ids),
            total,
        )

        for _ in range(initial):
            submit_one(next_index)
            next_index += 1

        while in_flight:
            if (
                stop_event is not None
                and stop_event.is_set()
            ):
                stopped = True

            for gpu_id, proc in zip(
                self.gpu_ids,
                self.procs,
            ):
                if proc.exitcode is not None:
                    raise RuntimeError(
                        f"GPU worker {gpu_id} died during generation "
                        f"with exit code {proc.exitcode}."
                    )

            try:
                msg = self.result_q.get(timeout=1)
            except pyqueue.Empty:
                continue

            if msg.get("batch_id") != batch_id:
                continue

            task_id = msg.get("task_id")

            if task_id not in in_flight:
                continue

            idx = in_flight.pop(task_id)

            if msg.get("status") != "ok":
                failing_text = ""

                if 0 <= idx < len(task_specs):
                    try:
                        failing_text = (
                            task_specs[idx]
                            .get("kwargs", {})
                            .get("text", "")
                        )
                    except Exception:
                        failing_text = ""

                raise RuntimeError(
                    f"GPU {msg.get('gpu')} generation failed "
                    f"on chunk {idx + 1}/{total}:\n"
                    f"{msg.get('error')}\n"
                    f"Failing text: {repr(str(failing_text)[:240])}\n\n"
                    f"{msg.get('traceback', '')}"
                )

            results.append(msg)

            if on_result is not None:
                try:
                    on_result(
                        msg,
                        len(results),
                        total,
                    )
                except Exception as callback_error:
                    print(
                        "⚠ Preview callback error:",
                        repr(callback_error),
                    )

            if (
                not stopped
                and next_index < total
            ):
                submit_one(next_index)
                next_index += 1

        return (
            sorted(
                results,
                key=lambda r: r.get(
                    "chunk_index",
                    0,
                ),
            ),
            stopped,
        )

    def shutdown(
        self,
        force=False,
    ):
        for _ in self.procs:
            try:
                self.task_q.put_nowait(
                    None
                )
            except Exception:
                pass

        for p in self.procs:
            try:
                p.join(
                    timeout=5
                )
            except Exception:
                pass

            if (
                force
                or p.is_alive()
            ):
                try:
                    p.terminate()
                except Exception:
                    pass


# Close a previous Cell-4 service if this cell is rerun.
if "_VOX_SERVICE" in globals():
    try:
        _VOX_SERVICE.shutdown(
            force=True
        )
    except Exception:
        pass

print(
    "Loading one VoxCPM2 "
    "replica on each T4..."
)

_VOX_SERVICE = DualGPUVoxCPM(
    MODEL_DIR,
    gpu_ids=(0, 1),
    optimize=WORKER_OPTIMIZE,
    startup_timeout=(
        WORKER_STARTUP_TIMEOUT
    ),
)

print(
    "✓ Dual-T4 service ready"
)

# ------------------------------------------------------------
# DATASET SYNC
# ------------------------------------------------------------

def sync_library_to_dataset(
    handle,
):
    """
    Sync only permanent metadata + permanent reference WAVs.
    Generated audio, imports and cookies are excluded.
    """
    handle = (
        handle or ""
    ).strip()

    if "/" not in handle:
        raise ValueError(
            "Dataset handle must "
            "look like "
            "username/dataset-slug"
        )

    import kagglehub

    sync_dir = Path(
        "/kaggle/working/"
        "voxcpm_voice_dataset_sync"
    )

    shutil.rmtree(
        sync_dir,
        ignore_errors=True,
    )

    (
        sync_dir
        / "voices"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    if META_FILE.exists():
        shutil.copy2(
            META_FILE,
            sync_dir
            / "library.json",
        )

    for vid in valid_voice_ids():
        src = voice_path(
            vid
        )

        shutil.copy2(
            src,
            sync_dir
            / "voices"
            / src.name,
        )

    kagglehub.dataset_upload(
        handle,
        str(sync_dir),
        version_notes=(
            "Voice Studio sync "
            + time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        ),
    )

# ------------------------------------------------------------
# UI STATE
# ------------------------------------------------------------

STATE = {
    "audition_path": None,
    "audition_text": None,
    "audition_direction": None,
    "audition_profile_id": None,

    "clone_path": None,
    "clone_transcript": "",
    "clone_source": "",

    "clone_busy": False,
    "audition_busy": False,
    "generation_busy": False,

    "cookie_file": None,
    "stop_event": None,
}

# ------------------------------------------------------------
# CSS / HEADER
# ------------------------------------------------------------

studio_css = widgets.HTML(
    r"""
<style>
.vox-card {
    background: #181818;
    color: #ffffff;
    border: 1px solid #343434;
    border-radius: 14px;
    padding: 16px 18px;
    margin: 8px 0 14px 0;
}
.vox-title {
    font-size: 27px;
    font-weight: 760;
    color: #ffffff;
}
.vox-sub {
    color: #aaaaaa;
    font-size: 13px;
    margin-top: 5px;
}
.vox-section {
    color: inherit;
    font-size: 18px;
    font-weight: 700;
    margin: 10px 0 8px 0;
}
.vox-note {
    color: #999999;
    font-size: 12px;
    line-height: 1.55;
}
.widget-button {
    border-radius: 10px !important;
    font-weight: 650 !important;
}
.widget-textarea textarea,
.widget-text input,
.widget-dropdown select {
    border-radius: 9px !important;
}
</style>
"""
)

header = widgets.HTML(
    r"""
<div class="vox-card">
  <div class="vox-title">
    🎙️ VoxCPM2 Voice Studio
  </div>
  <div class="vox-sub">
    Dual T4 • Burmese long-form • live chunk streaming • voice design • cloning
  </div>
</div>
"""
)

# ------------------------------------------------------------
# SHARED VOICE SELECTOR
# ------------------------------------------------------------

saved_voice = widgets.Dropdown(
    description="Voice",
    layout=widgets.Layout(
        width="560px"
    ),
)

play_selected_btn = widgets.Button(
    description="▶ Play",
    layout=widgets.Layout(
        width="95px"
    ),
)

delete_selected_btn = widgets.Button(
    description="🗑 Delete",
    button_style="danger",
    layout=widgets.Layout(
        width="110px"
    ),
)

library_output = widgets.Output()


def selected_voice_id():
    return (
        saved_voice.value
        or ""
    )


def refresh_saved_voices(
    select_id=None,
):
    ids = valid_voice_ids()

    ids.sort(
        key=lambda vid: (
            library[
                "voices"
            ][vid].get(
                "created",
                "",
            )
        ),
        reverse=True,
    )

    if ids:
        saved_voice.options = [
            (
                (
                    f"{library['voices'][vid].get('name', vid)}"
                    f"   •   "
                    f"{library['voices'][vid].get('created', '')}"
                ),
                vid,
            )
            for vid in ids
        ]

        desired = (
            select_id
            or library.get(
                "active_voice_id"
            )
            or ids[0]
        )

        saved_voice.value = (
            desired
            if desired in ids
            else ids[0]
        )

        play_selected_btn.disabled = False
        delete_selected_btn.disabled = False

    else:
        saved_voice.options = [
            (
                "— no saved voice yet —",
                "",
            )
        ]

        saved_voice.value = ""
        play_selected_btn.disabled = True
        delete_selected_btn.disabled = True


def on_voice_change(
    change=None,
):
    vid = selected_voice_id()

    if (
        vid
        and vid
        in library["voices"]
    ):
        library[
            "active_voice_id"
        ] = vid

        save_library()


saved_voice.observe(
    on_voice_change,
    names="value",
)


def on_play_selected(_):
    library_output.clear_output(
        wait=True
    )

    vid = selected_voice_id()

    if not vid:
        library_output.append_stdout(
            "No voice selected.\n"
        )
        return

    entry = (
        library["voices"][vid]
    )

    path = voice_path(
        vid
    )

    library_output.append_stdout(
        f"{entry.get('name', vid)}\n"
        f"Source: "
        f"{entry.get('source_type', 'saved voice')}\n\n"
    )

    library_output.append_display_data(
        Audio(
            filename=str(path),
            autoplay=False,
        )
    )


def on_delete_selected(_):
    library_output.clear_output(
        wait=True
    )

    if STATE[
        "generation_busy"
    ]:
        library_output.append_stdout(
            "Generation is running. "
            "Delete the voice after "
            "generation finishes.\n"
        )
        return

    vid = selected_voice_id()

    if not vid:
        return

    entry = (
        library["voices"]
        .get(
            vid,
            {},
        )
    )

    p = (
        LIBRARY_ROOT
        / entry.get(
            "file",
            "",
        )
    )

    try:
        if p.exists():
            p.unlink()
    except Exception:
        pass

    library["voices"].pop(
        vid,
        None,
    )

    if (
        library.get(
            "active_voice_id"
        )
        == vid
    ):
        library[
            "active_voice_id"
        ] = None

    save_library()
    refresh_saved_voices()

    library_output.append_stdout(
        "✓ Voice deleted "
        "from working library.\n"
    )


play_selected_btn.on_click(
    on_play_selected
)

delete_selected_btn.on_click(
    on_delete_selected
)

# ------------------------------------------------------------
# NOTEBOOK PLAYER / DOWNLOAD HELPERS
# ------------------------------------------------------------

def notebook_file_href(path):
    """
    Extract the notebook-served URL used by IPython FileLink.
    """
    path = str(path)

    try:
        rendered = FileLink(path)._repr_html_()

        match = re.search(
            r"href=['\"]([^'\"]+)['\"]",
            rendered,
        )

        if match:
            return match.group(1)

    except Exception:
        pass

    return path


def make_static_audio_card(
    path,
    title="Audio",
    mime="audio/wav",
):
    """
    Short-file player for profile tests / clone references.
    Includes seconds, three-dot menu, playback speed and download.
    """
    path = Path(path)

    player_id = (
        "vox_static_"
        + uuid.uuid4().hex[:10]
    )

    menu_id = player_id + "_menu"
    audio_id = player_id + "_audio"

    audio_src = file_as_data_url(
        path,
        mime=mime,
    )

    download_href = notebook_file_href(
        path
    )

    html = f"""
<div class="vox-card" style="padding:14px; position:relative;">
  <div style="
      display:flex;
      align-items:center;
      justify-content:space-between;
      gap:10px;
      margin-bottom:8px;
  ">
    <div style="font-weight:700;">{title}</div>

    <div style="position:relative;">
      <button
        onclick="
          const m=document.getElementById('{menu_id}');
          m.style.display=(m.style.display==='block'?'none':'block');
        "
        style="
          border:0;
          background:transparent;
          color:inherit;
          font-size:22px;
          cursor:pointer;
          line-height:1;
        "
        title="More"
      >⋯</button>

      <div
        id="{menu_id}"
        style="
          display:none;
          position:absolute;
          right:0;
          top:28px;
          z-index:9999;
          min-width:180px;
          padding:10px;
          border-radius:10px;
          background:#222;
          color:#fff;
          box-shadow:0 6px 24px rgba(0,0,0,.35);
        "
      >
        <div style="font-size:12px; margin-bottom:5px;">
          Playback speed
        </div>

        <select
          onchange="
            document.getElementById('{audio_id}').playbackRate =
              parseFloat(this.value);
          "
          style="width:100%; margin-bottom:10px;"
        >
          <option value="0.75">0.75×</option>
          <option value="1" selected>1.00×</option>
          <option value="1.25">1.25×</option>
          <option value="1.5">1.50×</option>
          <option value="2">2.00×</option>
        </select>

        <a
          href="{download_href}"
          target="_blank"
          download
          style="
            display:block;
            color:#fff;
            text-decoration:none;
            padding:7px 0;
          "
        >
          ⬇ Download
        </a>
      </div>
    </div>
  </div>

  <audio
    id="{audio_id}"
    src="{audio_src}"
    controls
    preload="metadata"
    style="width:100%;"
  ></audio>
</div>
"""

    return HTML(html)


# ------------------------------------------------------------
# GENERATE TAB — CONTROLS
# ------------------------------------------------------------

gen_script = widgets.Textarea(
    value="",
    placeholder=(
        "Paste your Burmese "
        "or English script here..."
    ),
    layout=widgets.Layout(
        width="100%",
        height="300px",
    ),
)

delivery_preset = widgets.Dropdown(
    options=list(
        DELIVERY_PRESETS.keys()
    ),
    value=(
        "Natural / "
        "keep cloned delivery"
    ),
    description="Delivery",
    layout=widgets.Layout(
        width="560px"
    ),
)

custom_style = widgets.Text(
    value="",
    description="Custom",
    placeholder=(
        "tone/emotion only — "
        "e.g. calm, tense, cheerful"
    ),
    layout=widgets.Layout(
        width="100%"
    ),
)

speed_slider = widgets.FloatSlider(
    value=1.00,
    min=0.75,
    max=1.50,
    step=0.05,
    description="Speed",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

chunk_chars = widgets.IntSlider(
    value=220,
    min=120,
    max=360,
    step=10,
    description="Chunk chars",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

join_pause = widgets.IntSlider(
    value=120,
    min=0,
    max=1000,
    step=20,
    description="Join pause",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

trim_edges_checkbox = widgets.Checkbox(
    value=True,
    description=(
        "Trim generated edge silence "
        "before joining"
    ),
    indent=False,
)

trim_db = widgets.FloatSlider(
    value=-42.0,
    min=-60.0,
    max=-25.0,
    step=1.0,
    description="Silence dB",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

cfg_slider = widgets.FloatSlider(
    value=2.0,
    min=1.0,
    max=3.0,
    step=0.1,
    description="CFG",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

steps_slider = widgets.IntSlider(
    value=10,
    min=4,
    max=30,
    step=1,
    description="Steps",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

generate_btn = widgets.Button(
    description="⚡ Generate + Stream",
    button_style="success",
    layout=widgets.Layout(
        width="285px",
        height="50px",
    ),
)

gen_status = widgets.HTML()

gen_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="Progress",
    bar_style="info",
    orientation="horizontal",
    layout=widgets.Layout(
        width="100%"
    ),
)

gen_text_note = widgets.HTML(
    "<div class='vox-note'>"
    "<b>Burmese-safe mode:</b> "
    "VoxCPM WeText normalization is OFF. "
    "Empty/punctuation-only chunks are filtered automatically."
    "</div>"
)

gen_stream_note = widgets.HTML(
    "<div class='vox-note'>"
    "<b>One live player only:</b> completed chunks are queued behind the "
    "audio you are already hearing. New chunks do <b>not</b> replace the "
    "current audio source, so they do not restart playback. If you catch "
    "up with generation, the player shows <b>Buffering — still generating</b>."
    "</div>"
)

generate_btn = widgets.Button(
    description="⚡ Generate + Stream",
    button_style="success",
    layout=widgets.Layout(
        width="285px",
        height="50px",
    ),
)

stop_btn = widgets.Button(
    description="■ Stop generating",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(
        width="190px",
        height="50px",
    ),
)

stream_player_output = widgets.Output()

STREAM_PLAYER_ID = (
    "voxcpm_stream_"
    + uuid.uuid4().hex[:10]
)

STREAM_STATUS_ID = STREAM_PLAYER_ID + "_status"
STREAM_TIME_ID = STREAM_PLAYER_ID + "_time"
STREAM_MENU_ID = STREAM_PLAYER_ID + "_menu"
STREAM_DOWNLOAD_ID = STREAM_PLAYER_ID + "_download"


def _stream_player_shell():
    return HTML(
        f"""
<div class="vox-card" style="padding:14px; position:relative;">
  <div style="
      display:flex;
      justify-content:space-between;
      align-items:center;
      gap:12px;
      margin-bottom:8px;
  ">
    <div style="font-weight:700;">
      🔴 Live / Final Voiceover
    </div>

    <div style="position:relative;">
      <button
        onclick="
          const m=document.getElementById('{STREAM_MENU_ID}');
          m.style.display=(m.style.display==='block'?'none':'block');
        "
        style="
          border:0;
          background:transparent;
          color:inherit;
          font-size:24px;
          cursor:pointer;
          line-height:1;
        "
        title="More"
      >⋯</button>

      <div
        id="{STREAM_MENU_ID}"
        style="
          display:none;
          position:absolute;
          right:0;
          top:28px;
          z-index:9999;
          width:200px;
          padding:11px;
          border-radius:10px;
          background:#222;
          color:#fff;
          box-shadow:0 6px 24px rgba(0,0,0,.35);
        "
      >
        <div style="font-size:12px; margin-bottom:5px;">
          Playback speed
        </div>

        <select
          id="{STREAM_PLAYER_ID}_speed"
          style="width:100%; margin-bottom:10px;"
        >
          <option value="0.75">0.75×</option>
          <option value="1" selected>1.00×</option>
          <option value="1.25">1.25×</option>
          <option value="1.5">1.50×</option>
          <option value="2">2.00×</option>
        </select>

        <a
          id="{STREAM_DOWNLOAD_ID}"
          href="#"
          style="
            display:block;
            color:#777;
            pointer-events:none;
            text-decoration:none;
            padding:7px 0;
          "
        >
          ⬇ Download final WAV
        </a>
      </div>
    </div>
  </div>

  <audio
    id="{STREAM_PLAYER_ID}"
    controls
    preload="auto"
    style="width:100%;"
  ></audio>

  <div
    id="{STREAM_TIME_ID}"
    style="
      margin-top:7px;
      font-size:12px;
      font-variant-numeric:tabular-nums;
      color:#bdbdbd;
    "
  >
    00:00 / 00:00 generated
  </div>

  <div
    id="{STREAM_STATUS_ID}"
    style="
      margin-top:5px;
      color:#bdbdbd;
      font-size:12px;
    "
  >
    Waiting for the first generated audio…
  </div>
</div>
"""
    )


def _init_stream_player(
    total_chunks,
):
    stream_player_output.clear_output(
        wait=True
    )

    stream_player_output.append_display_data(
        _stream_player_shell()
    )

    js = Javascript(
        f"""
(() => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const sid = {json.dumps(STREAM_STATUS_ID)};
  const tid = {json.dumps(STREAM_TIME_ID)};
  const speedId = {json.dumps(STREAM_PLAYER_ID + "_speed")};
  const total = {int(total_chunks)};

  window.__voxStreamV7 =
    window.__voxStreamV7 || {{}};

  const audio = document.getElementById(id);
  const status = document.getElementById(sid);
  const timeEl = document.getElementById(tid);
  const speedEl = document.getElementById(speedId);

  if (!audio) return;

  const state = {{
    segments: {{}},
    durations: {{}},
    current: -1,
    total: total,
    finalCount: null,
    generatedCount: 0,
    generationDone: false,
    stopped: false,
    wantPlay: false,
    buffering: false,
    switching: false,
    rate: 1.0
  }};

  window.__voxStreamV7[id] = state;

  const fmt = (sec) => {{
    sec = Math.max(0, Math.floor(Number(sec) || 0));
    const m = Math.floor(sec / 60);
    const s = sec % 60;
    return `${{String(m).padStart(2,'0')}}:${{String(s).padStart(2,'0')}}`;
  }};

  const contiguousCount = () => {{
    let i = 0;
    while (state.segments[i]) i++;
    return i;
  }};

  const availableSeconds = () => {{
    const count = contiguousCount();
    let totalSec = 0;

    for (let i = 0; i < count; i++) {{
      totalSec += Number(state.durations[i] || 0);
    }}

    return totalSec;
  }};

  const secondsBefore = (idx) => {{
    let totalSec = 0;

    for (let i = 0; i < idx; i++) {{
      totalSec += Number(state.durations[i] || 0);
    }}

    return totalSec;
  }};

  const updateTime = () => {{
    const currentBase =
      state.current >= 0
        ? secondsBefore(state.current)
        : 0;

    const played =
      currentBase
      + (Number(audio.currentTime) || 0);

    const available =
      availableSeconds();

    if (timeEl) {{
      timeEl.textContent =
        `${{fmt(played)}} / ${{fmt(available)}} generated`;
    }}
  }};

  const setStatus = (message) => {{
    if (status) status.textContent = message;
  }};

  const loadIndex = async (idx, autoplay) => {{
    const src = state.segments[idx];

    if (!src) {{
      state.buffering = true;

      setStatus(
        `⏳ Buffering — still generating… ` +
        `(${{state.generatedCount}}/${{state.total}} chunks ready)`
      );

      return false;
    }}

    state.switching = true;
    state.buffering = false;
    state.current = idx;

    // Critical behavior:
    // incoming chunks NEVER replace the source currently being played.
    audio.src = src;
    audio.playbackRate = state.rate;
    audio.load();

    const startPlayback = async () => {{
      state.switching = false;
      audio.playbackRate = state.rate;
      updateTime();

      if (autoplay) {{
        try {{
          await audio.play();
          state.wantPlay = true;

          setStatus(
            `▶ Playing • ${{state.generatedCount}}/${{state.total}} chunks generated`
          );
        }} catch (_) {{
          setStatus(
            `${{state.generatedCount}}/${{state.total}} chunks ready • press Play`
          );
        }}
      }} else {{
        setStatus(
          `${{state.generatedCount}}/${{state.total}} chunks ready • press Play`
        );
      }}
    }};

    if (audio.readyState >= 1) {{
      await startPlayback();
    }} else {{
      audio.addEventListener(
        "loadedmetadata",
        startPlayback,
        {{ once: true }}
      );
    }}

    return true;
  }};

  state.loadIndex = loadIndex;
  state.updateTime = updateTime;

  if (speedEl) {{
    speedEl.addEventListener("change", () => {{
      state.rate = parseFloat(speedEl.value || "1");
      audio.playbackRate = state.rate;
    }});
  }}

  audio.addEventListener("play", () => {{
    state.wantPlay = true;
    state.buffering = false;

    setStatus(
      `▶ Playing • ${{state.generatedCount}}/${{state.total}} chunks generated`
    );
  }});

  audio.addEventListener("pause", () => {{
    if (state.switching) return;

    if (!audio.ended) {{
      state.wantPlay = false;

      setStatus(
        `Paused • ${{state.generatedCount}}/${{state.total}} chunks generated`
      );
    }}
  }});

  audio.addEventListener("timeupdate", updateTime);
  audio.addEventListener("loadedmetadata", updateTime);

  audio.addEventListener("ended", async () => {{
    updateTime();

    const next = state.current + 1;

    const limit =
      state.generationDone && state.finalCount !== null
        ? state.finalCount
        : state.total;

    if (next >= limit) {{
      state.wantPlay = false;
      state.buffering = false;

      setStatus(
        state.stopped
          ? "■ Stopped • partial audio complete"
          : "✓ Playback complete"
      );

      return;
    }}

    state.wantPlay = true;

    if (state.segments[next]) {{
      await loadIndex(next, true);
    }} else {{
      state.buffering = true;

      setStatus(
        `⏳ Buffering — still generating… ` +
        `(${{state.generatedCount}}/${{state.total}} chunks ready)`
      );
    }}
  }});

  updateTime();
}})();
"""
    )

    stream_player_output.append_display_data(
        js
    )


def _queue_stream_segment(
    index,
    total_chunks,
    data_url,
    duration_seconds,
    contiguous_count,
):
    js = Javascript(
        f"""
(async () => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const sid = {json.dumps(STREAM_STATUS_ID)};

  const idx = {int(index)};
  const total = {int(total_chunks)};
  const src = {json.dumps(data_url)};
  const duration = {float(duration_seconds)};
  const contiguous = {int(contiguous_count)};

  const state =
    (window.__voxStreamV7 || {{}})[id];

  const audio =
    document.getElementById(id);

  const status =
    document.getElementById(sid);

  if (!state || !audio) return;

  // Queue only. Do NOT touch current audio.src.
  state.segments[idx] = src;
  state.durations[idx] = duration;
  state.generatedCount = contiguous;

  if (state.updateTime) {{
    state.updateTime();
  }}

  // First segment: attach once so the Play button becomes active.
  if (state.current < 0 && idx === 0) {{
    await state.loadIndex(0, false);
    return;
  }}

  // If playback already caught up and is buffering,
  // start the exact next queued segment.
  if (
    state.buffering &&
    idx === state.current + 1
  ) {{
    const shouldPlay = state.wantPlay;
    await state.loadIndex(idx, shouldPlay);
    return;
  }}

  if (status && !state.wantPlay) {{
    status.textContent =
      `${{contiguous}}/${{total}} chunks ready`;
  }}
}})();
"""
    )

    stream_player_output.append_display_data(
        js
    )


def _finish_stream_player(
    download_href,
    download_name,
    final_count,
    total_chunks,
    stopped=False,
):
    status_text = (
        "■ Generation stopped • partial WAV ready"
        if stopped
        else "✓ 100% generated • final WAV ready"
    )

    js = Javascript(
        f"""
(() => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const sid = {json.dumps(STREAM_STATUS_ID)};
  const did = {json.dumps(STREAM_DOWNLOAD_ID)};

  const state =
    (window.__voxStreamV7 || {{}})[id];

  const status =
    document.getElementById(sid);

  const download =
    document.getElementById(did);

  if (state) {{
    state.generationDone = true;
    state.finalCount = {int(final_count)};
    state.generatedCount = {int(final_count)};
    state.stopped = {str(bool(stopped)).lower()};

    if (state.updateTime) {{
      state.updateTime();
    }}
  }}

  if (download) {{
    download.href = {json.dumps(download_href)};
    download.download = {json.dumps(download_name)};
    download.style.color = "#fff";
    download.style.pointerEvents = "auto";
  }}

  if (status) {{
    status.textContent = {json.dumps(status_text)};
  }}
}})();
"""
    )

    stream_player_output.append_display_data(
        js
    )


def _fail_stream_player(
    message,
    final_count=0,
):
    js = Javascript(
        f"""
(() => {{
  const id = {json.dumps(STREAM_PLAYER_ID)};
  const sid = {json.dumps(STREAM_STATUS_ID)};

  const state =
    (window.__voxStreamV7 || {{}})[id];

  const status =
    document.getElementById(sid);

  if (state) {{
    state.generationDone = true;
    state.finalCount = {int(final_count)};
  }}

  if (status) {{
    status.textContent =
      {json.dumps("Generation stopped with error: " + str(message))};
  }}
}})();
"""
    )

    stream_player_output.append_display_data(
        js
    )


def build_delivery_control():
    base = (
        DELIVERY_PRESETS
        .get(
            delivery_preset.value,
            "",
        )
        .strip()
    )

    custom = (
        custom_style.value
        .strip()
    )

    return ", ".join(
        [
            x
            for x
            in (
                base,
                custom,
            )
            if x
        ]
    )


def make_generation_kwargs(
    chunk,
    ref_path,
    reference_text,
    control,
):
    common = {
        "cfg_value": float(
            cfg_slider.value
        ),
        "inference_timesteps": int(
            steps_slider.value
        ),
        "retry_badcase": True,
        "normalize": False,
    }

    if control:
        common.update(
            {
                "text": (
                    f"({control})"
                    f"{chunk}"
                ),
                "reference_wav_path": str(
                    ref_path
                ),
            }
        )

    else:
        common.update(
            {
                "text": chunk,
                "reference_wav_path": str(
                    ref_path
                ),
            }
        )

        if reference_text:
            common.update(
                {
                    "prompt_wav_path": str(
                        ref_path
                    ),
                    "prompt_text": (
                        reference_text
                    ),
                }
            )

    return common


def _build_downloadable_wav(
    prepared,
    count,
    stamp,
    stopped=False,
    errored=False,
):
    if count <= 0:
        return None, 0, 0

    ordered = [
        prepared[i]
        for i
        in range(count)
    ]

    if errored:
        name = (
            f"voiceover_partial_error_"
            f"{stamp}.wav"
        )
    elif stopped:
        name = (
            f"voiceover_stopped_"
            f"{stamp}.wav"
        )
    else:
        name = (
            f"voiceover_"
            f"{stamp}.wav"
        )

    final_path = (
        GENERATED_DIR
        / name
    )

    _, sr, samples = (
        concatenate_prepared_chunks(
            prepared_paths=ordered,
            output_path=final_path,
            pause_ms=int(
                join_pause.value
            ),
        )
    )

    return (
        final_path,
        sr,
        samples,
    )


def _generate_long_job(
    stop_event,
):
    prepared = {}
    next_stream_index = 0

    stamp = time.strftime(
        "%Y%m%d_%H%M%S"
    )

    total_chunks = 0

    try:
        vid = selected_voice_id()

        if not vid:
            raise ValueError(
                "Save/select a permanent "
                "voice first."
            )

        script = sanitize_tts_text(
            gen_script.value
        )

        if not script:
            raise ValueError(
                "Paste a script first."
            )

        entry = (
            library["voices"][vid]
        )

        ref = voice_path(
            vid
        )

        if not ref.exists():
            raise FileNotFoundError(
                "Selected reference "
                "WAV is missing."
            )

        reference_text = (
            entry
            .get(
                "transcript",
                "",
            )
            or ""
        ).strip()

        chunks = split_long_text(
            script,
            max_chars=int(
                chunk_chars.value
            ),
        )

        chunks = clean_chunks_for_tts(
            chunks
        )

        if not chunks:
            raise ValueError(
                "No speakable text chunks "
                "were created."
            )

        total_chunks = len(
            chunks
        )

        control = (
            build_delivery_control()
        )

        batch_dir = (
            GENERATED_DIR
            / (
                f"batch_{stamp}_"
                f"{uuid.uuid4().hex[:6]}"
            )
        )

        batch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        stream_dir = (
            batch_dir
            / "stream_ready"
        )

        stream_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        gen_progress.min = 0
        gen_progress.max = total_chunks
        gen_progress.value = 0
        gen_progress.bar_style = "info"

        _init_stream_player(
            total_chunks
        )

        gen_status.value = (
            f"<b>Generating…</b> "
            f"0/{total_chunks} chunks • "
            f"two T4 workers • "
            f"{entry.get('name', vid)}"
        )

        tasks = []

        for i, chunk in enumerate(
            chunks
        ):
            out = (
                batch_dir
                / f"chunk_{i:04d}.wav"
            )

            tasks.append(
                {
                    "chunk_index": i,
                    "output_path": str(
                        out
                    ),
                    "kwargs": (
                        make_generation_kwargs(
                            chunk=chunk,
                            ref_path=ref,
                            reference_text=reference_text,
                            control=control,
                        )
                    ),
                }
            )

        arrived = {}
        callback_lock = (
            threading.Lock()
        )

        def on_chunk_finished(
            msg,
            finished_count,
            total_count,
        ):
            nonlocal next_stream_index

            idx = int(
                msg["chunk_index"]
            )

            with callback_lock:
                arrived[idx] = msg

                gen_progress.value = int(
                    finished_count
                )

                while (
                    next_stream_index
                    in arrived
                ):
                    ready = (
                        arrived[
                            next_stream_index
                        ]
                    )

                    part_path = (
                        stream_dir
                        / (
                            f"part_"
                            f"{next_stream_index:04d}"
                            f".wav"
                        )
                    )

                    _, sr, sample_count = (
                        prepare_stream_chunk(
                            raw_path=(
                                ready[
                                    "output_path"
                                ]
                            ),
                            output_path=(
                                part_path
                            ),
                            speed=float(
                                speed_slider.value
                            ),
                            trim_edges=bool(
                                trim_edges_checkbox.value
                            ),
                            trim_db=float(
                                trim_db.value
                            ),
                        )
                    )

                    prepared[
                        next_stream_index
                    ] = str(
                        part_path
                    )

                    stream_mp3 = (
                        stream_dir
                        / (
                            f"stream_"
                            f"{next_stream_index:04d}"
                            f".mp3"
                        )
                    )

                    make_stream_segment_mp3(
                        prepared_wav=(
                            part_path
                        ),
                        output_mp3=(
                            stream_mp3
                        ),
                        pause_ms=int(
                            join_pause.value
                        ),
                        add_pause=True,
                    )

                    audio_seconds = (
                        float(sample_count)
                        / float(sr)
                    )

                    stream_duration = (
                        audio_seconds
                        + (
                            float(
                                join_pause.value
                            )
                            / 1000.0
                        )
                    )

                    _queue_stream_segment(
                        index=(
                            next_stream_index
                        ),
                        total_chunks=(
                            total_count
                        ),
                        data_url=(
                            file_as_data_url(
                                stream_mp3,
                                mime="audio/mpeg",
                            )
                        ),
                        duration_seconds=(
                            stream_duration
                        ),
                        contiguous_count=(
                            next_stream_index
                            + 1
                        ),
                    )

                    next_stream_index += 1

                gen_status.value = (
                    f"<b>Generating…</b> "
                    f"{finished_count}/"
                    f"{total_count} worker "
                    f"tasks finished • "
                    f"{next_stream_index}/"
                    f"{total_count} streamable"
                )

        start = time.time()

        results, stopped = (
            _VOX_SERVICE.run(
                tasks,
                on_result=(
                    on_chunk_finished
                ),
                stop_event=(
                    stop_event
                ),
            )
        )

        completed_count = (
            next_stream_index
        )

        if completed_count <= 0:
            if stopped:
                gen_status.value = (
                    "<b>■ Stopped</b> "
                    "before any audio finished."
                )

                _fail_stream_player(
                    "Stopped before first chunk",
                    final_count=0,
                )

                return

            raise RuntimeError(
                "Generation ended without "
                "a completed audio chunk."
            )

        final_path, sr, samples = (
            _build_downloadable_wav(
                prepared=prepared,
                count=completed_count,
                stamp=stamp,
                stopped=stopped,
            )
        )

        elapsed = (
            time.time()
            - start
        )

        duration = (
            samples / sr
            if sr
            else 0
        )

        _finish_stream_player(
            download_href=(
                notebook_file_href(
                    final_path
                )
            ),
            download_name=(
                final_path.name
            ),
            final_count=(
                completed_count
            ),
            total_chunks=(
                total_chunks
            ),
            stopped=(
                stopped
            ),
        )

        if stopped:
            gen_progress.value = (
                completed_count
            )

            gen_progress.bar_style = (
                "warning"
            )

            gen_status.value = (
                f"<b>■ Stopped</b> • "
                f"{completed_count}/"
                f"{total_chunks} chunks saved • "
                f"{duration:.1f}s partial audio"
            )

        else:
            gen_progress.value = (
                total_chunks
            )

            gen_progress.bar_style = (
                "success"
            )

            used = sorted(
                {
                    r["gpu"]
                    for r
                    in results
                }
            )

            gen_status.value = (
                f"<b>✓ Finished</b> • "
                f"{elapsed:.1f}s processing • "
                f"{duration:.1f}s audio • "
                f"GPUs {used}"
            )

    except Exception as e:
        completed_count = (
            next_stream_index
        )

        if completed_count > 0:
            try:
                partial_path, _, _ = (
                    _build_downloadable_wav(
                        prepared=prepared,
                        count=completed_count,
                        stamp=stamp,
                        errored=True,
                    )
                )

                _finish_stream_player(
                    download_href=(
                        notebook_file_href(
                            partial_path
                        )
                    ),
                    download_name=(
                        partial_path.name
                    ),
                    final_count=(
                        completed_count
                    ),
                    total_chunks=(
                        total_chunks
                        or completed_count
                    ),
                    stopped=True,
                )

            except Exception:
                _fail_stream_player(
                    str(e),
                    final_count=(
                        completed_count
                    ),
                )

        else:
            _fail_stream_player(
                str(e),
                final_count=0,
            )

        gen_progress.bar_style = (
            "danger"
        )

        gen_status.value = (
            "<span style='color:#ff5f56'>"
            "<b>Generation failed:</b> "
            f"{str(e)}"
            "</span>"
        )

    finally:
        STATE[
            "generation_busy"
        ] = False

        STATE[
            "stop_event"
        ] = None

        generate_btn.disabled = False

        stop_btn.disabled = True
        stop_btn.description = (
            "■ Stop generating"
        )


def on_generate(_):
    if STATE[
        "generation_busy"
    ]:
        return

    stop_event = (
        threading.Event()
    )

    STATE[
        "stop_event"
    ] = stop_event

    STATE[
        "generation_busy"
    ] = True

    generate_btn.disabled = True
    stop_btn.disabled = False
    stop_btn.description = (
        "■ Stop generating"
    )

    threading.Thread(
        target=_generate_long_job,
        args=(
            stop_event,
        ),
        daemon=True,
    ).start()


def on_stop_generation(_):
    event = STATE.get(
        "stop_event"
    )

    if (
        not STATE.get(
            "generation_busy"
        )
        or event is None
    ):
        return

    event.set()

    stop_btn.disabled = True
    stop_btn.description = (
        "Stopping…"
    )

    gen_status.value = (
        "<b>Stopping…</b> "
        "No new chunks will start. "
        "The one or two chunks already on the GPUs "
        "will finish, then the partial WAV will be saved."
    )


generate_btn.on_click(
    on_generate
)

stop_btn.on_click(
    on_stop_generation
)

generate_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                chunk_chars,
                join_pause,
                trim_edges_checkbox,
                trim_db,
                cfg_slider,
                steps_slider,
            ]
        )
    ],
    selected_index=None,
)

generate_advanced.set_title(
    0,
    "Advanced generation / chunk settings",
)

generate_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Voice"
            "</div>"
        ),
        widgets.HBox(
            [
                saved_voice,
                play_selected_btn,
            ]
        ),
        widgets.HTML(
            '<div class="vox-section">'
            "Script"
            "</div>"
        ),
        gen_script,
        widgets.HTML(
            '<div class="vox-section">'
            "Voice settings"
            "</div>"
        ),
        delivery_preset,
        custom_style,
        speed_slider,
        widgets.HTML(
            '<div class="vox-note">'
            "<b>Generation Speed</b> changes the saved audio itself. "
            "The ⋯ menu on the player changes playback speed only."
            "</div>"
        ),
        generate_advanced,
        gen_text_note,
        widgets.HBox(
            [
                generate_btn,
                stop_btn,
            ],
            layout=widgets.Layout(
                gap="10px",
                flex_flow="row wrap",
            ),
        ),
        gen_status,
        gen_progress,
        gen_stream_note,
        stream_player_output,
    ]
)

# ------------------------------------------------------------
# VOICE DESIGN TAB
# ------------------------------------------------------------

profile_select = widgets.Dropdown(
    options=[
        (
            p["name"],
            pid,
        )
        for pid, p
        in VOICE_PROFILES.items()
    ],
    value="u_aung",
    description="Profile",
    layout=widgets.Layout(
        width="640px"
    ),
)

profile_info = widgets.HTML()

design_audition_text = widgets.Textarea(
    value=(
        "မင်္ဂလာပါ။ "
        "ဒီအသံက သဘာဝကျပြီး "
        "နားထောင်ရလွယ်ကူတဲ့ "
        "မြန်မာစကားပြောသံ "
        "ဖြစ်စေချင်ပါတယ်။"
    ),
    placeholder=(
        "Use one to three "
        "sentences..."
    ),
    layout=widgets.Layout(
        width="100%",
        height="120px",
    ),
)

design_name = widgets.Text(
    value="My Burmese Voice",
    description="Save name",
    layout=widgets.Layout(
        width="640px"
    ),
)

design_btn = widgets.Button(
    description=(
        "🎲 Generate profile candidate"
    ),
    button_style="success",
    layout=widgets.Layout(
        width="280px",
        height="46px",
    ),
)

save_design_btn = widgets.Button(
    description="🔒 Save candidate",
    button_style="warning",
    disabled=True,
    layout=widgets.Layout(
        width="220px",
        height="46px",
    ),
)

design_status = widgets.HTML()
design_output = widgets.Output()

save_design_btn.description = (
    "💾 Save this voice"
)

save_design_btn.layout = widgets.Layout(
    width="220px",
    height="44px",
)

save_design_btn.disabled = True

design_player_box = widgets.VBox(
    [
        design_output,
        save_design_btn,
    ]
)


def refresh_profile(
    change=None,
):
    profile = (
        VOICE_PROFILES[
            profile_select.value
        ]
    )

    profile_info.value = (
        "<div class='vox-card'>"
        f"<b>{profile['name']}</b>"
        "<br>"
        "<span class='vox-note'>"
        f"{profile['design']}"
        "</span>"
        "</div>"
    )


profile_select.observe(
    refresh_profile,
    names="value",
)

refresh_profile()


def _audition_job():
    try:
        pid = (
            profile_select.value
        )

        profile = (
            VOICE_PROFILES[pid]
        )

        text = sanitize_tts_text(
            design_audition_text.value
        )

        if not has_speakable_content(
            text
        ):
            raise ValueError(
                "Enter a short "
                "audition script."
            )

        direction = profile[
            "design"
        ]

        out = (
            GENERATED_DIR
            / (
                f"audition_"
                f"{pid}_"
                f"{int(time.time())}_"
                f"{uuid.uuid4().hex[:5]}"
                f".wav"
            )
        )

        design_status.value = (
            f"<b>Generating:</b> "
            f"{profile['name']}"
        )

        results, _ = (
            _VOX_SERVICE.run(
                [
                    {
                        "chunk_index": 0,
                        "output_path": str(
                            out
                        ),
                        "kwargs": {
                            "text": (
                                f"({direction})"
                                f"{text}"
                            ),
                            "cfg_value": 2.0,
                            "inference_timesteps": 10,
                            "retry_badcase": True,
                            "normalize": False,
                        },
                    }
                ]
            )
        )

        result = results[0]

        STATE[
            "audition_path"
        ] = str(out)

        STATE[
            "audition_text"
        ] = text

        STATE[
            "audition_direction"
        ] = direction

        STATE[
            "audition_profile_id"
        ] = pid

        design_status.value = (
            f"<b>✓ Candidate ready</b> "
            f"• GPU "
            f"{result.get('gpu')}"
        )

        design_output.clear_output(
            wait=True
        )

        design_output.append_display_data(
            make_static_audio_card(
                path=out,
                title=(
                    "🎧 Test voice candidate"
                ),
                mime="audio/wav",
            )
        )

        save_design_btn.disabled = False

    except Exception as e:
        design_status.value = (
            "<span style='color:#ff5f56'>"
            "<b>Failed:</b> "
            f"{str(e)}"
            "</span>"
        )

    finally:
        STATE[
            "audition_busy"
        ] = False

        design_btn.disabled = False


def on_design(_):
    if STATE[
        "audition_busy"
    ]:
        return

    STATE[
        "audition_busy"
    ] = True

    design_btn.disabled = True
    save_design_btn.disabled = True
    save_design_btn.description = (
        "💾 Save this voice"
    )

    design_output.clear_output(
        wait=True
    )

    threading.Thread(
        target=_audition_job,
        daemon=True,
    ).start()


def on_save_design(_):
    candidate = STATE.get(
        "audition_path"
    )

    if (
        not candidate
        or not Path(
            candidate
        ).exists()
    ):
        design_status.value = (
            "<span style='color:#ff5f56'>"
            "Generate a candidate first."
            "</span>"
        )
        return

    name = (
        design_name.value.strip()
        or "Designed Voice"
    )

    vid = (
        uuid.uuid4().hex[:12]
    )

    dest = (
        VOICE_DIR
        / f"{vid}.wav"
    )

    shutil.copy2(
        candidate,
        dest,
    )

    library[
        "voices"
    ][vid] = {
        "name": name,
        "file": (
            f"voices/{vid}.wav"
        ),
        "transcript": (
            STATE.get(
                "audition_text"
            )
            or ""
        ),
        "direction": (
            STATE.get(
                "audition_direction"
            )
            or ""
        ),
        "profile_id": (
            STATE.get(
                "audition_profile_id"
            )
        ),
        "source_type": (
            "voice_design"
        ),
        "created": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    library[
        "active_voice_id"
    ] = vid

    save_library()

    refresh_saved_voices(
        select_id=vid
    )

    save_design_btn.disabled = True
    save_design_btn.description = (
        "✓ Saved"
    )

    design_status.value = (
        f"<b>🔒 Saved:</b> "
        f"{name}"
    )


design_btn.on_click(
    on_design
)

save_design_btn.on_click(
    on_save_design
)

design_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Choose a test voice profile"
            "</div>"
        ),
        profile_select,
        profile_info,
        widgets.HTML(
            '<div class="vox-section">'
            "Short audition text"
            "</div>"
        ),
        design_audition_text,
        design_name,
        design_btn,
        design_status,
        design_player_box,
    ]
)

# ------------------------------------------------------------
# VOICE CLONE TAB
# ------------------------------------------------------------

clone_source_mode = widgets.ToggleButtons(
    options=[
        (
            "Upload file",
            "upload",
        ),
        (
            "Remote URL",
            "url",
        ),
    ],
    value="upload",
    description="Source",
)

clone_upload = widgets.FileUpload(
    accept=(
        ".wav,.mp3,.m4a,.aac,.flac,"
        ".ogg,.opus,.wma,"
        ".mp4,.mkv,.webm,.mov,.avi,"
        ".mpeg,.mpg,.m4v"
    ),
    multiple=False,
    description="Choose audio/video",
)

clone_url = widgets.Text(
    value="",
    placeholder=(
        "YouTube / TikTok / "
        "Google Drive / MEGA / "
        "direct media URL"
    ),
    layout=widgets.Layout(
        width="100%"
    ),
)

cookie_upload = widgets.FileUpload(
    accept=".txt",
    multiple=False,
    description="cookies.txt",
)

save_cookie_btn = widgets.Button(
    description="Use cookie file",
    layout=widgets.Layout(
        width="150px"
    ),
)

clear_cookie_btn = widgets.Button(
    description="Clear cookies",
    layout=widgets.Layout(
        width="125px"
    ),
)

cookie_status = widgets.HTML()

remote_user_agent = widgets.Text(
    value="",
    description="User-Agent",
    placeholder="optional",
    layout=widgets.Layout(
        width="100%"
    ),
)

remote_referer = widgets.Text(
    value="",
    description="Referer",
    placeholder="optional",
    layout=widgets.Layout(
        width="100%"
    ),
)

extract_vocals = widgets.Checkbox(
    value=False,
    description=(
        "Remove music / SFX "
        "with Demucs vocal isolation"
    ),
    indent=False,
)

reduce_noise = widgets.Checkbox(
    value=True,
    description=(
        "Reduce residual "
        "background noise"
    ),
    indent=False,
)

noise_amount = widgets.FloatSlider(
    value=0.70,
    min=0.20,
    max=1.00,
    step=0.05,
    description="Noise remove",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

clone_start = widgets.FloatText(
    value=0.0,
    description="Start sec",
    layout=widgets.Layout(
        width="280px"
    ),
)

clone_max = widgets.FloatSlider(
    value=20.0,
    min=5.0,
    max=45.0,
    step=1.0,
    description="Max seconds",
    continuous_update=False,
    layout=widgets.Layout(
        width="560px"
    ),
)

whisper_model = widgets.Dropdown(
    options=[
        (
            "Whisper large-v3-turbo",
            "large-v3-turbo",
        ),
        (
            "Whisper large-v3",
            "large-v3",
        ),
    ],
    value="large-v3-turbo",
    description="ASR model",
    layout=widgets.Layout(
        width="620px"
    ),
)

whisper_language = widgets.Dropdown(
    options=[
        (
            "Burmese / Myanmar",
            "my",
        ),
        (
            "Auto-detect",
            "auto",
        ),
        (
            "English",
            "en",
        ),
    ],
    value="my",
    description="Language",
    layout=widgets.Layout(
        width="520px"
    ),
)

clone_name = widgets.Text(
    value="My Cloned Voice",
    description="Voice name",
    layout=widgets.Layout(
        width="640px"
    ),
)

process_clone_btn = widgets.Button(
    description=(
        "✨ Prepare & transcribe clone"
    ),
    button_style="success",
    layout=widgets.Layout(
        width="290px",
        height="48px",
    ),
)

save_clone_btn = widgets.Button(
    description=(
        "🔒 Save clone to library"
    ),
    button_style="warning",
    disabled=True,
    layout=widgets.Layout(
        width="250px",
        height="48px",
    ),
)

clone_status = widgets.HTML()

clone_transcript = widgets.Textarea(
    value="",
    placeholder=(
        "Whisper transcript appears here. "
        "Correct it before saving if necessary."
    ),
    layout=widgets.Layout(
        width="100%",
        height="150px",
    ),
)

clone_output = widgets.Output()


def on_save_cookie(_):
    try:
        path = save_cookie_upload(
            cookie_upload
        )

        if not path:
            raise ValueError(
                "Choose a Netscape "
                "cookies.txt file first."
            )

        STATE[
            "cookie_file"
        ] = path

        cookie_status.value = (
            "<span style='color:#2ea043'>"
            "✓ Cookie file loaded "
            "for this session."
            "</span>"
        )

    except Exception as e:
        cookie_status.value = (
            "<span style='color:#ff5f56'>"
            f"{str(e)}"
            "</span>"
        )


def on_clear_cookie(_):
    STATE[
        "cookie_file"
    ] = None

    path = (
        COOKIE_DIR
        / "cookies.txt"
    )

    if path.exists():
        try:
            path.unlink()
        except Exception:
            pass

    cookie_status.value = (
        "Cookies cleared."
    )


save_cookie_btn.on_click(
    on_save_cookie
)

clear_cookie_btn.on_click(
    on_clear_cookie
)


def set_clone_status(
    text,
    color="#8c8c8c",
):
    clone_status.value = (
        "<div class='vox-card' "
        f"style='color:{color}'>"
        f"{text}"
        "</div>"
    )


def _clone_pipeline():
    job_dir = (
        IMPORT_DIR
        / (
            f"clone_"
            f"{int(time.time())}_"
            f"{uuid.uuid4().hex[:6]}"
        )
    )

    job_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        if (
            clone_source_mode.value
            == "upload"
        ):
            set_clone_status(
                "1/5 • Reading uploaded media…"
            )

            source = save_upload_to_dir(
                clone_upload,
                job_dir,
                fallback_name=(
                    "source.bin"
                ),
            )

        else:
            set_clone_status(
                "1/5 • Downloading remote media…"
            )

            source = download_remote_media(
                clone_url.value,
                job_dir,
                cookie_file=(
                    STATE.get(
                        "cookie_file"
                    )
                ),
                user_agent=(
                    remote_user_agent.value
                ),
                referer=(
                    remote_referer.value
                ),
            )

        STATE[
            "clone_source"
        ] = str(source)

        set_clone_status(
            "2/5 • Converting to mono WAV "
            "and selecting reference segment…"
        )

        normalized = (
            job_dir
            / "reference_normalized.wav"
        )

        normalize_media_to_wav(
            source_path=source,
            output_path=normalized,
            start_seconds=max(
                0.0,
                float(
                    clone_start.value
                ),
            ),
            max_seconds=max(
                1.0,
                float(
                    clone_max.value
                ),
            ),
        )

        working = str(
            normalized
        )

        if extract_vocals.value:
            set_clone_status(
                "3/5 • Removing music / SFX "
                "with Demucs on CPU…"
            )

            try:
                working = extract_vocals_demucs(
                    working,
                    job_dir,
                )

            except Exception as e:
                set_clone_status(
                    "Demucs failed; continuing "
                    "with normalized source. "
                    f"{str(e)}",
                    color="#b7791f",
                )

                working = str(
                    normalized
                )

        else:
            set_clone_status(
                "3/5 • Music/SFX "
                "isolation skipped."
            )

        if reduce_noise.value:
            set_clone_status(
                "4/5 • Reducing residual "
                "background noise…"
            )

            denoised = (
                job_dir
                / "denoised.wav"
            )

            reduce_residual_noise(
                working,
                denoised,
                amount=float(
                    noise_amount.value
                ),
            )

            working = str(
                denoised
            )

        final_ref = (
            job_dir
            / "clone_reference.wav"
        )

        normalize_media_to_wav(
            source_path=working,
            output_path=final_ref,
            start_seconds=0,
            max_seconds=max(
                1.0,
                float(
                    clone_max.value
                ),
            ),
        )

        wav, sr = sf.read(
            str(final_ref),
            dtype="float32",
            always_2d=False,
        )

        wav = trim_edge_silence(
            wav,
            sr,
            threshold_db=-45.0,
            keep_ms=40,
        )

        sf.write(
            str(final_ref),
            wav,
            int(sr),
            subtype="PCM_16",
        )

        set_clone_status(
            f"5/5 • Transcribing with "
            f"{whisper_model.value} "
            f"on CPU…"
        )

        (
            transcript,
            detected,
        ) = transcribe_reference(
            final_ref,
            model_name=(
                whisper_model.value
            ),
            language=(
                whisper_language.value
            ),
        )

        STATE[
            "clone_path"
        ] = str(
            final_ref
        )

        STATE[
            "clone_transcript"
        ] = transcript

        clone_transcript.value = (
            transcript
        )

        clone_output.clear_output(
            wait=True
        )

        clone_output.append_stdout(
            "✓ Prepared clone reference\n"
            f"Detected language: "
            f"{detected or 'unknown'}\n\n"
        )

        clone_output.append_display_data(
            make_static_audio_card(
                path=final_ref,
                title="🧬 Prepared clone reference",
                mime="audio/wav",
            )
        )

        save_clone_btn.disabled = False

        set_clone_status(
            "✓ Clone reference is ready. "
            "Listen, correct the transcript "
            "if needed, then save it.",
            color="#2ea043",
        )

    except Exception as e:
        set_clone_status(
            "<b>Clone preparation failed:</b> "
            f"{str(e)}",
            color="#ff5f56",
        )

    finally:
        STATE[
            "clone_busy"
        ] = False

        process_clone_btn.disabled = False


def on_process_clone(_):
    if STATE[
        "clone_busy"
    ]:
        return

    STATE[
        "clone_busy"
    ] = True

    process_clone_btn.disabled = True
    save_clone_btn.disabled = True

    threading.Thread(
        target=_clone_pipeline,
        daemon=True,
    ).start()


def on_save_clone(_):
    ref = STATE.get(
        "clone_path"
    )

    if (
        not ref
        or not Path(
            ref
        ).exists()
    ):
        set_clone_status(
            "Prepare a clone "
            "reference first.",
            color="#ff5f56",
        )
        return

    transcript = (
        clone_transcript.value
        .strip()
    )

    if not transcript:
        set_clone_status(
            "Transcript is empty. "
            "Enter the exact spoken words "
            "before saving.",
            color="#b7791f",
        )
        return

    name = (
        clone_name.value.strip()
        or "Cloned Voice"
    )

    vid = (
        uuid.uuid4().hex[:12]
    )

    dest = (
        VOICE_DIR
        / f"{vid}.wav"
    )

    shutil.copy2(
        ref,
        dest,
    )

    library[
        "voices"
    ][vid] = {
        "name": name,
        "file": (
            f"voices/{vid}.wav"
        ),
        "transcript": transcript,
        "direction": "",
        "profile_id": None,
        "source_type": (
            "voice_clone"
        ),
        "source": (
            STATE.get(
                "clone_source",
                "",
            )
        ),
        "created": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    library[
        "active_voice_id"
    ] = vid

    save_library()
    refresh_saved_voices(
        select_id=vid
    )

    set_clone_status(
        f"🔒 Saved cloned voice: "
        f"{name}",
        color="#2ea043",
    )


process_clone_btn.on_click(
    on_process_clone
)

save_clone_btn.on_click(
    on_save_clone
)

cookie_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                widgets.HTML(
                    "<div class='vox-note'>"
                    "Optional for sites requiring "
                    "your own authenticated session. "
                    "Upload a Netscape-format cookies.txt. "
                    "Do not share the cookie file."
                    "</div>"
                ),
                widgets.HBox(
                    [
                        cookie_upload,
                        save_cookie_btn,
                        clear_cookie_btn,
                    ]
                ),
                cookie_status,
                remote_user_agent,
                remote_referer,
            ]
        )
    ],
    selected_index=None,
)

cookie_advanced.set_title(
    0,
    "Advanced remote-download cookies / headers",
)

cleanup_advanced = widgets.Accordion(
    children=[
        widgets.VBox(
            [
                extract_vocals,
                reduce_noise,
                noise_amount,
                clone_start,
                clone_max,
                whisper_model,
                whisper_language,
            ]
        )
    ],
    selected_index=None,
)

cleanup_advanced.set_title(
    0,
    "Cleanup / transcription settings",
)

clone_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Clone a voice from audio/video"
            "</div>"
        ),
        widgets.HTML(
            '<div class="vox-note">'
            "Use media you have permission to access and clone. "
            "A clean single-speaker sample works best."
            "</div>"
        ),
        clone_source_mode,
        widgets.HTML(
            "<b>Direct upload</b>"
        ),
        clone_upload,
        widgets.HTML(
            "<b>Remote URL</b>"
        ),
        clone_url,
        cookie_advanced,
        cleanup_advanced,
        clone_name,
        widgets.HBox(
            [
                process_clone_btn,
                save_clone_btn,
            ]
        ),
        clone_status,
        widgets.HTML(
            '<div class="vox-section">'
            "Reference transcript"
            "</div>"
        ),
        clone_transcript,
        clone_output,
    ]
)

# ------------------------------------------------------------
# LIBRARY / PERSISTENCE TAB
# ------------------------------------------------------------

dataset_handle = widgets.Text(
    value=(
        VOICE_DATASET_HANDLE_DEFAULT
    ),
    description="Dataset",
    placeholder=(
        "username/dataset-slug"
    ),
    layout=widgets.Layout(
        width="640px"
    ),
)

sync_btn = widgets.Button(
    description=(
        "☁ Sync voice library"
    ),
    button_style="info",
    layout=widgets.Layout(
        width="220px",
        height="44px",
    ),
)

sync_status = widgets.HTML()


def on_sync(_):
    sync_btn.disabled = True

    try:
        sync_status.value = (
            "Uploading current "
            "library…"
        )

        sync_library_to_dataset(
            dataset_handle.value
        )

        sync_status.value = (
            "<span style='color:#2ea043'>"
            "✓ Kaggle Dataset synced. "
            "Attach its latest version "
            "as notebook Input next time."
            "</span>"
        )

    except Exception as e:
        sync_status.value = (
            "<span style='color:#ff5f56'>"
            "Sync failed: "
            f"{str(e)}"
            "</span>"
        )

    finally:
        sync_btn.disabled = False


sync_btn.on_click(
    on_sync
)

library_tab = widgets.VBox(
    [
        widgets.HTML(
            '<div class="vox-section">'
            "Permanent voice library"
            "</div>"
        ),
        widgets.HBox(
            [
                saved_voice,
                play_selected_btn,
                delete_selected_btn,
            ]
        ),
        library_output,
        widgets.HTML(
            '<div class="vox-section">'
            "Persistent Kaggle Dataset"
            "</div>"
        ),
        widgets.HTML(
            '<div class="vox-note">'
            "Sync library.json and permanent voice WAVs "
            "to a Kaggle Dataset. Generated files, imported "
            "media and cookies are not uploaded."
            "</div>"
        ),
        dataset_handle,
        sync_btn,
        sync_status,
    ]
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

refresh_saved_voices()

if restored_from:
    sync_status.value = (
        "<span style='color:#2ea043'>"
        "✓ Restored voice library from: "
        f"{restored_from}"
        "</span>"
    )

tabs = widgets.Tab(
    children=[
        generate_tab,
        design_tab,
        clone_tab,
        library_tab,
    ],
    layout=widgets.Layout(
        width="100%"
    ),
)

tabs.set_title(
    0,
    "⚡ Generate",
)

tabs.set_title(
    1,
    "🎭 Voice Design",
)

tabs.set_title(
    2,
    "🧬 Voice Clone",
)

tabs.set_title(
    3,
    "💾 Library",
)

display(
    studio_css
)

display(
    header
)

display(
    tabs
)


Loading one VoxCPM2 replica on each T4...
Starting 2 VoxCPM2 workers (optimize=False)...
  • worker PID 161 → physical GPU 0
  • worker PID 162 → physical GPU 1
  … still loading GPU worker(s) 0, 1
  … still loading GPU worker(s) 0, 1
  • GPU 1 is loading VoxCPM2...
  • GPU 0 is loading VoxCPM2...


voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False
voxcpm_model_path: /kaggle/working/VoxCPM2, zipenhancer_model_path: None, enable_denoiser: False
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Loading AudioVAE from pytorch: /kaggle/working/VoxCPM2/audiovae.pth
Running on device: cuda, dtype: bfloat16
Running on device: cuda, dtype: bfloat16


  … still loading GPU worker(s) 0, 1
  … still loading GPU worker(s) 0, 1


Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors
Loading model from safetensors: /kaggle/working/VoxCPM2/model.safetensors


  … still loading GPU worker(s) 0, 1


Loaded VoxCPM2Model
Loaded VoxCPM2Model


✓ GPU 0 ready (48000 Hz)
✓ GPU 1 ready (48000 Hz)
✓ Both VoxCPM2 workers are ready.
✓ Dual-T4 service ready


HTML(value='\n<style>\n.vox-card {\n    background: #181818;\n    color: #ffffff;\n    border: 1px solid #3434…

HTML(value='\n<div class="vox-card">\n  <div class="vox-title">\n    🎙️ VoxCPM2 Voice Studio\n  </div>\n  <div…

## Notes

- **Style fix:** when a delivery instruction is present, the studio uses VoxCPM2 reference-audio-only controllable cloning instead of mixing the style instruction with prompt-text continuation. This prevents text such as “slightly faster” from being treated like narration.
- **Speed:** the Speed control uses FFmpeg `atempo`, so it is deterministic and preserves pitch.
- **Join pause:** generated edge silence is trimmed first; then the requested pause is inserted, so 100 ms/300 ms now has a predictable effect.
- **Clone preprocessing:** Demucs and Whisper run on CPU in a background thread because both T4s are reserved for VoxCPM2.
- Use only media you have permission to download and clone. Cookies stay in a separate private working folder and are never included by the voice-library sync. Imported source media and generated outputs are also excluded from the persistent Dataset.


### VoxCPM package compatibility
This revision inspects `model._generate` inside each GPU worker. Unsupported keyword arguments are removed automatically. On older builds without native `seed=`, a requested deterministic seed is applied directly to Python, NumPy and PyTorch RNGs instead.


### Progressive / live listening
Long generation now exposes completed chunks immediately. Because a normal WAV file is finalized with its length in the header and Kaggle/Jupyter rich output is served as a static object, the studio uses **ordered segmented streaming**: Part 1 appears as soon as it finishes, then Part 2, Part 3, and so on while both GPUs keep synthesizing later chunks. The final combined WAV is still created at the end.


### Streaming player v4
The live player now keeps one cumulative preview. After each **contiguous** chunk is ready, the notebook rebuilds a small 64-kbps MP3 preview and updates the same browser audio player while preserving its playback position. If playback reaches the currently generated edge first, the UI shows **“Buffering — still generating…”**; when another chunk arrives, the player attempts to resume automatically. Individual completed chunks are also shown as fallback audio players. The final downloadable result remains a full-quality WAV.
